[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_05/NB_bishop_ch05_classification.ipynb)

# Chapter 5: Single-layer Networks -- Classification

This notebook implements classification models from Bishop's *Deep Learning: Foundations and Concepts*, Chapter 5: logistic regression from scratch with gradient descent, softmax for multi-class classification, ROC curves, and decision boundary visualization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from scipy import stats
from sklearn.datasets import make_moons, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, accuracy_score

matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

def save_fig(fig, name, chart_dir='../../charts'):
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            legend._ncols = min(len(legend.get_texts()), 4)
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)

## 1. Generate 2D Classification Data

In [ ]:
np.random.seed(42)

# Two linearly separable Gaussian clusters
N_per_class = 100
X_c0 = np.random.multivariate_normal([1, 1], [[0.8, 0.3], [0.3, 0.5]], N_per_class)
X_c1 = np.random.multivariate_normal([3, 3], [[0.6, -0.2], [-0.2, 0.8]], N_per_class)

X = np.vstack([X_c0, X_c1])
y = np.array([0] * N_per_class + [1] * N_per_class)

# Shuffle
idx = np.random.permutation(len(y))
X, y = X[idx], y[idx]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1],
           facecolors='none', edgecolors='#1a3a6e', s=40, label='Class 0')
ax.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1],
           facecolors='none', edgecolors='#cd0000', s=40, label='Class 1')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('2D Classification Data')
ax.legend()
fig.tight_layout()
plt.show()

## 2. Logistic Regression from Scratch

The logistic sigmoid: $\sigma(a) = \frac{1}{1 + e^{-a}}$

Model: $p(C_1 | \mathbf{x}) = \sigma(\mathbf{w}^T \boldsymbol{\phi}(\mathbf{x}))$

Cross-entropy loss:
$$E(\mathbf{w}) = -\sum_{n=1}^{N} \left[ t_n \ln y_n + (1 - t_n) \ln(1 - y_n) \right]$$

In [ ]:
def sigmoid(a):
    """Numerically stable sigmoid."""
    return np.where(a >= 0,
                    1 / (1 + np.exp(-a)),
                    np.exp(a) / (1 + np.exp(a)))

def add_bias(X):
    """Prepend a column of ones for the bias term."""
    return np.hstack([np.ones((X.shape[0], 1)), X])

def cross_entropy_loss(y_pred, y_true):
    """Binary cross-entropy."""
    eps = 1e-15
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

def logistic_gradient(X_b, y, w):
    """Gradient of cross-entropy loss w.r.t. weights."""
    y_pred = sigmoid(X_b @ w)
    return X_b.T @ (y_pred - y) / len(y)

def logistic_regression_gd(X, y, lr=0.1, n_iters=1000):
    """Train logistic regression with gradient descent."""
    X_b = add_bias(X)
    w = np.zeros(X_b.shape[1])
    losses = []

    for i in range(n_iters):
        y_pred = sigmoid(X_b @ w)
        loss = cross_entropy_loss(y_pred, y)
        losses.append(loss)
        grad = logistic_gradient(X_b, y, w)
        w -= lr * grad

    return w, losses

In [ ]:
w_lr, losses = logistic_regression_gd(X_train, y_train, lr=1.0, n_iters=500)

# Training accuracy
y_pred_train = (sigmoid(add_bias(X_train) @ w_lr) >= 0.5).astype(int)
y_pred_test = (sigmoid(add_bias(X_test) @ w_lr) >= 0.5).astype(int)

print(f'Training accuracy: {accuracy_score(y_train, y_pred_train):.4f}')
print(f'Test accuracy:     {accuracy_score(y_test, y_pred_test):.4f}')
print(f'Weights: w0={w_lr[0]:.4f}, w1={w_lr[1]:.4f}, w2={w_lr[2]:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(losses, color='#1a3a6e', linewidth=2)
ax.set_xlabel('Iteration')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('Logistic Regression: Training Loss')
fig.tight_layout()
plt.show()

## 3. Decision Boundary Visualization

In [ ]:
def plot_decision_boundary(X, y, w, ax, title='Decision Boundary'):
    """Plot data points and logistic regression decision boundary."""
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                          np.linspace(y_min, y_max, 300))
    grid = np.c_[xx.ravel(), yy.ravel()]
    grid_b = add_bias(grid)
    probs = sigmoid(grid_b @ w).reshape(xx.shape)

    ax.contourf(xx, yy, probs, levels=50, cmap='RdYlBu_r', alpha=0.4)
    ax.contour(xx, yy, probs, levels=[0.5], colors='black', linewidths=2)

    ax.scatter(X[y == 0, 0], X[y == 0, 1], facecolors='none',
               edgecolors='#1a3a6e', s=40, label='Class 0', zorder=5)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], facecolors='none',
               edgecolors='#cd0000', s=40, label='Class 1', zorder=5)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_title(title)
    ax.legend(fontsize=8)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

plot_decision_boundary(X_train, y_train, w_lr, ax1, 'Training Set')
plot_decision_boundary(X_test, y_test, w_lr, ax2, 'Test Set')

fig.suptitle('Bishop Ch 5: Logistic Regression Decision Boundary', fontsize=13, y=1.02)
fig.tight_layout()
save_fig(fig, 'bishop_ch5_logistic_boundary')
plt.show()

## 4. Effect of Learning Rate

In [ ]:
learning_rates = [0.01, 0.1, 1.0, 5.0]
colors_lr = ['#1a3a6e', '#cd0000', '#228B22', '#FF8C00']

fig, ax = plt.subplots(figsize=(8, 5))
for lr, c in zip(learning_rates, colors_lr):
    _, loss_hist = logistic_regression_gd(X_train, y_train, lr=lr, n_iters=300)
    ax.plot(loss_hist, color=c, linewidth=2, label=f'lr = {lr}')

ax.set_xlabel('Iteration')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('Convergence for Different Learning Rates')
ax.set_ylim(0, 1.0)
ax.legend()
fig.tight_layout()
plt.show()

## 5. Softmax Regression (Multi-class)

For $K$ classes:
$$p(C_k | \mathbf{x}) = \frac{\exp(\mathbf{w}_k^T \boldsymbol{\phi}(\mathbf{x}))}{\sum_{j=1}^{K} \exp(\mathbf{w}_j^T \boldsymbol{\phi}(\mathbf{x}))}$$

In [ ]:
def softmax(Z):
    """Numerically stable softmax."""
    Z_shifted = Z - Z.max(axis=1, keepdims=True)
    exp_Z = np.exp(Z_shifted)
    return exp_Z / exp_Z.sum(axis=1, keepdims=True)

def one_hot(y, K):
    """One-hot encoding."""
    N = len(y)
    T = np.zeros((N, K))
    T[np.arange(N), y] = 1
    return T

def softmax_regression_gd(X, y, K, lr=0.1, n_iters=500):
    """Train softmax regression with gradient descent."""
    X_b = add_bias(X)
    N, D = X_b.shape
    W = np.zeros((D, K))
    T = one_hot(y, K)
    losses = []

    for i in range(n_iters):
        Z = X_b @ W
        Y = softmax(Z)
        loss = -np.mean(np.sum(T * np.log(Y + 1e-15), axis=1))
        losses.append(loss)
        grad = X_b.T @ (Y - T) / N
        W -= lr * grad

    return W, losses

In [ ]:
# Generate 3-class data
np.random.seed(42)
X_multi, y_multi = make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=42)

X_m_train, X_m_test, y_m_train, y_m_test = train_test_split(
    X_multi, y_multi, test_size=0.3, random_state=42)

W_soft, losses_soft = softmax_regression_gd(X_m_train, y_m_train, K=3, lr=0.1, n_iters=500)

# Predictions
y_m_pred = np.argmax(softmax(add_bias(X_m_test) @ W_soft), axis=1)
print(f'Softmax regression test accuracy: {accuracy_score(y_m_test, y_m_pred):.4f}')

In [ ]:
# Decision regions for softmax
fig, ax = plt.subplots(figsize=(8, 6))

x_min, x_max = X_multi[:, 0].min() - 1, X_multi[:, 0].max() + 1
y_min, y_max = X_multi[:, 1].min() - 1, X_multi[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                      np.linspace(y_min, y_max, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
grid_b = add_bias(grid)
probs_grid = softmax(grid_b @ W_soft)
preds_grid = np.argmax(probs_grid, axis=1).reshape(xx.shape)

from matplotlib.colors import ListedColormap
cmap_bg = ListedColormap(['#d4e6f1', '#fadbd8', '#d5f5e3'])
colors_multi = ['#1a3a6e', '#cd0000', '#228B22']

ax.contourf(xx, yy, preds_grid, alpha=0.3, cmap=cmap_bg)

for k, c in enumerate(colors_multi):
    mask = y_m_test == k
    ax.scatter(X_m_test[mask, 0], X_m_test[mask, 1],
               facecolors='none', edgecolors=c, s=50, label=f'Class {k}', zorder=5)

ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('Bishop Ch 5: Softmax Regression Decision Regions')
ax.legend()
fig.tight_layout()
save_fig(fig, 'bishop_ch5_softmax')
plt.show()

## 6. ROC Curve and AUC

The Receiver Operating Characteristic curve plots True Positive Rate vs False Positive Rate at various thresholds.

In [ ]:
# Use the binary logistic model predictions
y_scores = sigmoid(add_bias(X_test) @ w_lr)

fpr, tpr, thresholds = roc_curve(y_test, y_scores)
roc_auc = auc(fpr, tpr)

print(f'AUC = {roc_auc:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

ax.plot(fpr, tpr, color='#cd0000', linewidth=2.5,
        label=f'Logistic Regression (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], '--', color='gray', linewidth=1.5, label='Random Classifier')
ax.fill_between(fpr, tpr, alpha=0.1, color='#cd0000')

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Bishop Ch 5: ROC Curve')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.set_aspect('equal')
ax.legend()
fig.tight_layout()
save_fig(fig, 'bishop_ch5_roc')
plt.show()

## 7. Effect of Threshold on Precision and Recall

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds_pr = precision_recall_curve(y_test, y_scores)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(recall, precision, color='#1a3a6e', linewidth=2)
ax.fill_between(recall, precision, alpha=0.1, color='#1a3a6e')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.05)
fig.tight_layout()
plt.show()

## 8. Sigmoid and Softmax Functions Visualized

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Sigmoid
a = np.linspace(-6, 6, 300)
ax1.plot(a, sigmoid(a), color='#1a3a6e', linewidth=2.5)
ax1.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
ax1.axvline(0, color='gray', linestyle=':', alpha=0.5)
ax1.set_xlabel('a')
ax1.set_ylabel(r'$\sigma(a)$')
ax1.set_title('Logistic Sigmoid')

# Softmax for 3 classes as function of a_1 (a_2 = 0, a_3 = 1)
a1_range = np.linspace(-4, 6, 300)
for k, (a2, a3, c, lbl) in enumerate([
    (0, 1, '#1a3a6e', '$p(C_1)$'),
    (0, 1, '#cd0000', '$p(C_2)$'),
    (0, 1, '#228B22', '$p(C_3)$')
]):
    a_all = np.column_stack([a1_range, np.full_like(a1_range, a2),
                             np.full_like(a1_range, a3)])
    probs_s = softmax(a_all)
    ax2.plot(a1_range, probs_s[:, k], color=c, linewidth=2, label=lbl)

ax2.set_xlabel('$a_1$')
ax2.set_ylabel('Probability')
ax2.set_title('Softmax ($a_2$=0, $a_3$=1)')
ax2.legend(fontsize=9)

fig.suptitle('Bishop Ch 5: Activation Functions', fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

## 9. Logistic Regression on Non-linear Data (Moon Dataset)

In [ ]:
X_moon, y_moon = make_moons(n_samples=300, noise=0.2, random_state=42)
X_moon_tr, X_moon_te, y_moon_tr, y_moon_te = train_test_split(
    X_moon, y_moon, test_size=0.3, random_state=42)

# Linear logistic regression
w_moon, _ = logistic_regression_gd(X_moon_tr, y_moon_tr, lr=1.0, n_iters=500)
y_moon_pred = (sigmoid(add_bias(X_moon_te) @ w_moon) >= 0.5).astype(int)
acc_linear = accuracy_score(y_moon_te, y_moon_pred)

# Add polynomial features for better fit
def poly_features(X):
    """Add quadratic features: x1^2, x2^2, x1*x2."""
    return np.column_stack([X, X[:, 0]**2, X[:, 1]**2, X[:, 0] * X[:, 1]])

X_moon_poly_tr = poly_features(X_moon_tr)
X_moon_poly_te = poly_features(X_moon_te)
w_moon_poly, _ = logistic_regression_gd(X_moon_poly_tr, y_moon_tr, lr=1.0, n_iters=1000)
y_moon_poly_pred = (sigmoid(add_bias(X_moon_poly_te) @ w_moon_poly) >= 0.5).astype(int)
acc_poly = accuracy_score(y_moon_te, y_moon_poly_pred)

print(f'Linear logistic accuracy: {acc_linear:.4f}')
print(f'Polynomial logistic accuracy: {acc_poly:.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Linear boundary
plot_decision_boundary(X_moon_te, y_moon_te, w_moon, ax1,
                        f'Linear (acc={acc_linear:.2f})')

# Polynomial boundary
x_min, x_max = X_moon[:, 0].min() - 0.5, X_moon[:, 0].max() + 0.5
y_min, y_max = X_moon[:, 1].min() - 0.5, X_moon[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                      np.linspace(y_min, y_max, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
grid_poly = poly_features(grid)
probs = sigmoid(add_bias(grid_poly) @ w_moon_poly).reshape(xx.shape)

ax2.contourf(xx, yy, probs, levels=50, cmap='RdYlBu_r', alpha=0.4)
ax2.contour(xx, yy, probs, levels=[0.5], colors='black', linewidths=2)
ax2.scatter(X_moon_te[y_moon_te == 0, 0], X_moon_te[y_moon_te == 0, 1],
            facecolors='none', edgecolors='#1a3a6e', s=40, label='Class 0', zorder=5)
ax2.scatter(X_moon_te[y_moon_te == 1, 0], X_moon_te[y_moon_te == 1, 1],
            facecolors='none', edgecolors='#cd0000', s=40, label='Class 1', zorder=5)
ax2.set_xlabel('$x_1$')
ax2.set_ylabel('$x_2$')
ax2.set_title(f'Polynomial Features (acc={acc_poly:.2f})')
ax2.legend(fontsize=8)

fig.suptitle('Bishop Ch 5: Linear vs Non-linear Decision Boundaries', fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

## 10. Summary

**Key takeaways from Bishop Chapter 5:**

- Logistic regression uses the sigmoid function to map linear combinations to probabilities.
- The cross-entropy loss is the appropriate objective for classification.
- Gradient descent iteratively updates weights in the direction of steepest descent.
- Softmax generalizes the sigmoid to multi-class problems.
- ROC curves and AUC provide threshold-independent evaluation of classifiers.
- Linear models can handle non-linear boundaries through feature engineering (basis functions).